Den här notebooken hanterar:

1. Inläsning av ads_clean.json
2. Skapande av ett “balanserat” testset med:
    - beskrivningar längre än 200 tecken (under den gränsen är lite eller ingen information relevant)
    - 3 grupper för beskrivningens längd (så att vi kontrollerar korta, medellånga och långa annonser)
    - 10 jobb per månad (jan–aug)
3. Sparande av det testsetet, redo att annoteras
4. Sparande av träningssetet

In [1]:
import pandas as pd
import json

with open("../data/ads_clean.json", encoding="utf-8") as f:
    df_clean = pd.DataFrame(json.load(f))

print("Antal rader:", len(df_clean))

Antal rader: 4264


# Test set

In [2]:
# checking ads with short descriptions
shorty = df_clean[df_clean["description"].str.len() < 200]

print("Antal korta beskrivningar:", len(shorty))
shorty

Antal korta beskrivningar: 7


,id,version_id,title,employer,municipality,region,occupation,published,deadline,description,language
65,29380853,80f8379915bbc4167ad20b83e1201c429d25fbfb,IT-konsulter,Careerguide Media Group AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-14T15:36:27,2025-01-17T23:59:59,Vi söker IT-konsulter för kunds räkning. Kvali...,sv
102,29390154,01d64bec03d9d1ce3b1242e6d4762e13a0f3c09a,IT-konsulter,Careerguide Media Group AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-17T13:48:02,2025-01-20T23:59:59,Vi söker IT-konsulter för kunds räkning. Kvali...,sv
123,29399546,ad3f2999f896170128d538311b19884db0f38ee3,IT-konsulter,Careerguide Media Group AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-21T16:34:57,2025-01-24T23:59:59,Vi söker IT-konsulter för kunds räkning. Kvali...,sv
383,29507839,2b4e9a743cc424f8ae8cd51e6bb580f14564a4af,Cloud- och Infrastrukturspecialist,Liminity AB,Stockholm,Stockholms län,Infrastrukturarkitekt,2025-03-01T15:00:49,2025-08-18T23:59:59,Som specialist kommer du att underhålla modern...,sv
940,29692048,247284cb1c8df6e32e3f202f0d4648c7c81a7445,Associate Principal - Architecture,"LTIMindtree Sverige, filial till LTIMindtree Lim",Stockholm,Stockholms län,Systemarkitekt,2025-05-09T14:54:59,2025-05-23T23:59:59,Vi är öppna för att anpassa rollen eller arbet...,sv
955,29700087,0c31e09ca9e253bc58c62a116848930cca53053e,Principal - Architecture,"LTIMindtree Sverige, filial till LTIMindtree Lim",Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2025-05-13T15:36:39,2025-06-12T23:59:59,Vi är öppna för att anpassa rollen eller arbet...,sv
1534,29920986,13f2864b257c1cc1ce2658a423ec4a2afeba1939,Pega Business Architect,BR Technologies AB,Stockholm,Stockholms län,Enterprise Architect,2025-08-18T17:14:50,2025-09-01T23:59:59,Vi söker en erfaren Pega Business Architect so...,sv


In [3]:
# remove jobs with descriptions shorter than 200 characters
df = df_clean[df_clean["description"].str.len() >= 200]

print("Antal rader efter borttagning av annonser med korta beskrivningar:", len(df))

Antal rader efter borttagning av annonser med korta beskrivningar: 4257


In [4]:
# restrict to the analysis window: Jan-Aug of both years
df["published"] = pd.to_datetime(df["published"])

df["year"] = df["published"].dt.year
df["month"] = df["published"].dt.month
df = df[(df["month"] <= 8) & (df["year"].isin([2025, 2026]))].copy()

print("Antal rader efter urval av relevanta månader:", len(df))
df.head()

Antal rader efter urval av relevanta månader: 3271


,id,version_id,title,employer,municipality,region,occupation,published,deadline,description,language,year,month
0,29354331,d127a2a996c37ab43c33c5607ad285fde7cbdc50,Payment Platform Manager,Spiruetta AB,Stockholm,Stockholms län,Systemarkitekt,2025-01-02 09:24:37,2025-02-01T23:59:59,Spiruetta AB söker en erfaren Payment Platform...,sv,2025,1
1,29354585,26008c8a15f486a5b9163029fddd5d7fc3c03cb4,Projektledare,Randstad AB,Nacka,Stockholms län,Infrastrukturarkitekt,2025-01-02 10:15:14,2025-01-27T23:59:59,"Arbetsbeskrivning\nVA, Projekt och utredning\n...",sv,2025,1
2,29355065,80fee065cdecb056026d75c18c095d96f64d2ec6,Consultant Manager till ett av våra grymma Dyn...,Fellowmind Sweden AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-02 11:39:23,2025-06-21T23:59:59,Att du dessutom får chansen att joina Sveriges...,sv,2025,1
3,29355172,ac28585698079a23f90482d302de7da2ccc8e1ed,Systemspecialist med inriktning IT-infrastrukt...,Trafikverket,Stockholm,Stockholms län,Systemanalytiker/Systemutredare,2025-01-02 12:07:23,2025-01-23T23:59:59,Är du intresserad av cybersäkerhet och vill va...,sv,2025,1
4,29355229,6bd58c9893a5bc98dc7adee09f49739847c31409,IT arkitekt / Lösningsarkitekt till energisekt...,Lexicare AB,Luleå,Norrbottens län,IT-arkitekt/Lösningsarkitekt,2025-01-02 12:17:37,2025-02-01T23:59:59,Uppdragsbeskrivning:\nLexicare AB söker en niv...,sv,2025,1


In [5]:
# length buckets
df["len_bucket"] = pd.qcut(df["description"].str.len(), 3, labels=["short", "medium", "long"])

# shuffles the ads
df = df.sample(frac=1, random_state=42)

df.head()

,id,version_id,title,employer,municipality,region,occupation,published,deadline,description,language,year,month,len_bucket
2637,30460963,790b55bb23f991c0a2bbc81ce7efe70ab5efd8e9,Expert Kravanalytiker,Professional Galaxy AB,Göteborg,Västra Götalands län,Kravanalytiker,2026-01-12 22:45:29,2026-07-11T23:59:59,Professional Galaxy är ett IT och teknik konsu...,sv,2026,1,medium
137,29406811,3d6f443794843836ff16ecf3a189519b7874ab75,Cloud Security Architect,Avaron AB,Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2025-01-23 18:36:07,2025-01-31T23:59:59,Om företaget\nAvaron AB är ett etablerat konsu...,sv,2025,1,short
415,29525446,f838a31c481fcee95e65eb25b913e7a4a9175bd9,"Sustainability Solution Architect, Stora Enso ...",Stora Enso Skog AB,Hammarö,Värmlands län,IT-arkitekt/Lösningsarkitekt,2025-03-10 08:03:58,2025-03-28T23:59:59,"Har du ett starkt intresse för hållbarhet, dig...",sv,2025,3,medium
206,29437136,d465ac3acdac862ff0e834c6ce6e93e260414926,Solutions Architect till Husqvarna Group!,Academic Work Sweden AB,Jönköping,Jönköpings län,IT-arkitekt/Lösningsarkitekt,2025-02-04 12:01:21,2025-03-05T23:59:59,Som Lösningsarkitekt på Husqvarna Group får du...,sv,2025,2,medium
1165,29758125,4ffd4b2f8b9b133982fa67b7138aa8388efac68e,Arkitekt inom IT-infrastruktur,Centric Professionals AB,Göteborg,Västra Götalands län,IT-arkitekt/Lösningsarkitekt,2025-06-09 14:16:23,2025-09-09T23:59:59,Nu söker vi en arkitekt inom IT-infrastruktur ...,sv,2025,6,medium


In [6]:
# helpfunction to select equal number of ads per month
def sample_per_month(d, n_per_month=10, seed=42):
    return d.groupby("month").sample(
        n=n_per_month, random_state=seed
    )

In [7]:
# creating the test set and checking it
test_set = pd.concat([sample_per_month(df[df.year == y]) for y in (2025, 2026)])

print("Antal anonser:", len(test_set))
print("-" * 50)
print("\nAntal anonser per månad och år:")
print(test_set.groupby(["year", "month"]).size().unstack(0))
print("-" * 50)
print("\nAntal annonser per längdgrupp och år:")
print(test_set.groupby(["year", "len_bucket"], observed=True).size().unstack(0))

Antal anonser: 160
--------------------------------------------------

Antal anonser per månad och år:
year   2025  2026
month            
1        10    10
2        10    10
3        10    10
4        10    10
5        10    10
6        10    10
7        10    10
8        10    10
--------------------------------------------------

Antal annonser per längdgrupp och år:
year        2025  2026
len_bucket            
short         26    31
medium        26    26
long          28    23


In [8]:
# The test set only need "id" and "description", so I drop the rest of the cols
test_set = test_set[["id", "description"]].reset_index(drop=True)
test_set.head()

,id,description
0,29395534,Arbetsuppgifter\nSom integrationskonsult hos o...
1,29355231,Uppdragsbeskrivning:\nLexicare AB söker en niv...
2,29381137,Vill du vara med och lösa en av de största sam...
3,29394109,Systemintegratör till Stockholm! Sigma Technol...
4,29367913,Q Professionals AB är ett företag nicschat mot...


In [9]:
# Saving test set
with open("../data/test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_set.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

# Training set

Träningssetet ska inte innehålla annonser från testsetet, så de tas bort och träningssetet sparas.

In [10]:
train_set = df_clean[~df_clean["id"].isin(test_set["id"])].reset_index(drop=True)

print("Antal annonser i hela den rensade dataframen:", len(df))
print("Antal annonser i test_set::", len(test_set))
print("Antal annonser i train_set::", len(train_set))
train_set.head()

Antal annonser i hela den rensade dataframen: 3271
Antal annonser i test_set:: 160
Antal annonser i train_set:: 4104


,id,version_id,title,employer,municipality,region,occupation,published,deadline,description,language
0,29354331,d127a2a996c37ab43c33c5607ad285fde7cbdc50,Payment Platform Manager,Spiruetta AB,Stockholm,Stockholms län,Systemarkitekt,2025-01-02T09:24:37,2025-02-01T23:59:59,Spiruetta AB söker en erfaren Payment Platform...,sv
1,29354585,26008c8a15f486a5b9163029fddd5d7fc3c03cb4,Projektledare,Randstad AB,Nacka,Stockholms län,Infrastrukturarkitekt,2025-01-02T10:15:14,2025-01-27T23:59:59,"Arbetsbeskrivning\nVA, Projekt och utredning\n...",sv
2,29355065,80fee065cdecb056026d75c18c095d96f64d2ec6,Consultant Manager till ett av våra grymma Dyn...,Fellowmind Sweden AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-02T11:39:23,2025-06-21T23:59:59,Att du dessutom får chansen att joina Sveriges...,sv
3,29355172,ac28585698079a23f90482d302de7da2ccc8e1ed,Systemspecialist med inriktning IT-infrastrukt...,Trafikverket,Stockholm,Stockholms län,Systemanalytiker/Systemutredare,2025-01-02T12:07:23,2025-01-23T23:59:59,Är du intresserad av cybersäkerhet och vill va...,sv
4,29355229,6bd58c9893a5bc98dc7adee09f49739847c31409,IT arkitekt / Lösningsarkitekt till energisekt...,Lexicare AB,Luleå,Norrbottens län,IT-arkitekt/Lösningsarkitekt,2025-01-02T12:17:37,2025-02-01T23:59:59,Uppdragsbeskrivning:\nLexicare AB söker en niv...,sv


In [11]:
# Saving test set
with open("../data/train_set.json", "w", encoding="utf-8") as f:
    json.dump(train_set.to_dict(orient="records"), f, ensure_ascii=False, indent=2)